In [56]:
import os
import json
import pandas as pd

def parse_output_file(file_path):
    print(file_path)
    result = {}
    with open(file_path, 'r') as file:
        for line in file.readlines():
            print("Line: ", line)
            line = line.strip()
            tokens = line.split(":")
            result[tokens[0].strip()] = tokens[1].strip()
    if 'Query throughput' in result:
        result['query_thput'] = float(result['Query throughput'].split()[0])
    if 'False positives' in result:
        result['fp'] = float(result['False positives'].split()[0])
    if 'False positive rate' in result:
        result['fpr'] = float(result['False positive rate'].split()[0][:-1]) / 100.0 # Ignore the '%' in the result
    if 'Filter throughput' in result:
        result['filter_thput'] = float(result['Filter throughput'].split()[0])
    if 'Filtered DB throughput' in result:
        result['db_thput'] = float(result['Filtered DB throughput'].split()[0])
    if 'ReverseMap DB throughput' in result:
        result['rm_thput'] = float(result['ReverseMap DB throughput'].split()[0])
    return result





In [59]:
result_dir = "./breakEven"
experiment_runs = []
for dir in os.listdir(result_dir):
    if dir == "_sources":
        continue
    experiment_run = {}
    experiment_run['id'] = str(dir)
    with open(os.path.join(result_dir, dir, 'config.json'), 'r', encoding='utf-8') as file:
        experiment_run['config'] = json.load(file)
    with open(os.path.join(result_dir, dir, 'run.json'), 'r', encoding='utf-8') as file:
        experiment_run['context'] = json.load(file)
    output_path = os.path.join(result_dir, dir, 'output.txt')
    if os.path.exists(output_path):
        experiment_run['result'] = parse_output_file(os.path.join(result_dir, dir, 'output.txt'))
        experiment_runs.append(experiment_run)

df = pd.json_normalize(experiment_runs)
df['result.breakeven'] = df['result.db_thput']/df['result.rm_thput']
print(df.columns)
display(df)

./breakEven/3/output.txt
Line:  Adaptive (filtered, batched) throughput:

Line:  Number of inserts:     3774873

Line:  Number of updates:     3774873

Line:  Time for inserts:      8.606484

Line:  Insert throughput:     438608.030875 ops/sec

Line:  CPU time for inserts:  8.418005

Line:  Time for queries:     1.557781 s

Line:  Filter throughput:     28483536.515894 ops/sec

Line:  Filtered DB throughput:     217442.215039 ops/sec

Line:  ReverseMap DB throughput:     15850.223523 ops/sec

Line:  Query throughput:     6419387.577586 ops/sec

Line:  False positives:      17827

Line:  Num DB Queries:      17827

Line:  False positive rate:  0.178270%

./breakEven/8/output.txt
Line:  ReverseMap throughput (Only NonEmpty Queries, No Filter):

Line:  bucketSize: 0 count:0

Line:  bucketSize: 1 count:3771751

Line:  bucketSize: 2 count:1561

Line:  Num Queries:      3774873

Line:  Time for queries:     103.919409 s

Line:  Query throughput:     36325.004504 ops/sec

Line:  CPU time for 

,id,config.filter,config.num_queries,config.quotient_bits,config.remainder_bits,config.seed,context.artifacts,context.command,context.experiment.base_dir,context.experiment.dependencies,...,result.fp,result.fpr,result.filter_thput,result.db_thput,result.rm_thput,"result.ReverseMap throughput (Only NonEmpty Queries, No Filter)",result.bucketSize,result.Num Queries,result.CPU time for queries,result.breakeven
0,3,adaptive,10000000,22,9,799178538,[output.txt],run_experiment,/home/chesetti/Repos/skiAdaptiveQf/bench,"[numpy==1.24.4, sacred==0.8.7]",...,17827.0,0.001783,2.848354e+07,217442.215039,15850.223523,NaN,NaN,NaN,NaN,13.718558
1,8,reverse,50000000,22,10,485039567,[output.txt],run_experiment,/home/chesetti/Repos/skiAdaptiveQf/bench,"[numpy==1.24.4, sacred==0.8.7]",...,NaN,0.075497,NaN,NaN,NaN,,2 count,3774873,37.728978 s,NaN
2,5,reverse,50000000,22,7,453009688,[output.txt],run_experiment,/home/chesetti/Repos/skiAdaptiveQf/bench,"[numpy==1.24.4, sacred==0.8.7]",...,NaN,0.075497,NaN,NaN,NaN,,3 count,3774873,38.191810 s,NaN
3,1,adaptive,10000000,22,7,93775130,[output.txt],run_experiment,/home/chesetti/Repos/skiAdaptiveQf/bench,"[numpy==1.24.4, sacred==0.8.7]",...,69937.0,0.006994,2.795373e+07,298555.822601,17157.565660,NaN,NaN,NaN,NaN,17.400826
4,2,adaptive,10000000,22,8,361703794,[output.txt],run_experiment,/home/chesetti/Repos/skiAdaptiveQf/bench,"[numpy==1.24.4, sacred==0.8.7]",...,35057.0,0.003506,2.799928e+07,241517.571115,15474.718564,NaN,NaN,NaN,NaN,15.607235
5,7,reverse,50000000,22,9,874904685,[output.txt],run_experiment,/home/chesetti/Repos/skiAdaptiveQf/bench,"[numpy==1.24.4, sacred==0.8.7]",...,NaN,0.075497,NaN,NaN,NaN,,3 count,3774873,37.645939 s,NaN
6,6,reverse,50000000,22,8,819209670,[output.txt],run_experiment,/home/chesetti/Repos/skiAdaptiveQf/bench,"[numpy==1.24.4, sacred==0.8.7]",...,NaN,0.075497,NaN,NaN,NaN,,3 count,3774873,36.592354 s,NaN
7,4,adaptive,10000000,22,10,433507648,[output.txt],run_experiment,/home/chesetti/Repos/skiAdaptiveQf/bench,"[numpy==1.24.4, sacred==0.8.7]",...,8777.0,0.000878,2.879330e+07,70206.451923,14817.153256,NaN,NaN,NaN,NaN,4.738188


### Breakeven cost


In [60]:
filtered_df = df[df['config.filter']=='adaptive']
display(filtered_df.groupby(['config.quotient_bits', 'config.remainder_bits', 'config.num_queries', 'config.filter'])
        .agg({
            'result.query_thput': ['mean'],
            'result.filter_thput': ['mean'],
            'result.fpr': ['mean'],
            'result.db_thput': ['mean'],
            'result.rm_thput': ['mean'],
            'result.breakeven': ['mean']
            }))

result.query_thput  \
                                                                                          mean   
config.quotient_bits config.remainder_bits config.num_queries config.filter                      
22                   7                     10000000           adaptive            2.142178e+06   
                     8                     10000000           adaptive            3.613054e+06   
                     9                     10000000           adaptive            6.419388e+06   
                     10                    10000000           adaptive            9.392546e+06   

                                                                            result.filter_thput  \
                                                                                           mean   
config.quotient_bits config.remainder_bits config.num_queries config.filter                       
22                   7                     10000000           adaptive             2.795373e+07   
                     8                     10000000           adaptive             2.799928e+07   
                     9                     10000000           adaptive             2.848354e+07   
                     10                    10000000           adaptive             2.879330e+07   

                                                                            result.fpr  \
                                                                                  mean   
config.quotient_bits config.remainder_bits config.num_queries config.filter              
22                   7                     10000000           adaptive        0.006994   
                     8                     10000000           adaptive        0.003506   
                     9                     10000000           adaptive        0.001783   
                     10                    10000000           adaptive        0.000878   

                                                                            result.db_thput  \
                                                                                       mean   
config.quotient_bits config.remainder_bits config.num_queries config.filter                   
22                   7                     10000000           adaptive        298555.822601   
                     8                     10000000           adaptive        241517.571115   
                     9                     10000000           adaptive        217442.215039   
                     10                    10000000           adaptive         70206.451923   

                                                                            result.rm_thput  \
                                                                                       mean   
config.quotient_bits config.remainder_bits config.num_queries config.filter                   
22                   7                     10000000           adaptive         17157.565660   
                     8                     10000000           adaptive         15474.718564   
                     9                     10000000           adaptive         15850.223523   
                     10                    10000000           adaptive         14817.153256   

                                                                            result.breakeven  
                                                                                        mean  
config.quotient_bits config.remainder_bits config.num_queries config.filter                   
22                   7                     10000000           adaptive             17.400826  
                     8                     10000000           adaptive             15.607235  
                     9                     10000000           adaptive             13.718558  
                     10                    10000000           adaptive              4.738188

### DB and ReverseMap Throughput

In [ ]:
filtered_df = df[(df['config.filter']=='reverse') | (df['config.filter']=='database')]
display(filtered_df.groupby(['config.quotient_bits', 'config.remainder_bits', 'config.num_queries', 'config.filter'])
        .agg({
            'result.query_thput': ['mean'],
            'result.filter_thput': ['mean'],
            'result.fpr': ['mean'],
            'result.db_thput': ['mean'],
            'result.rm_thput': ['mean'],
            'result.breakeven': ['mean']
            }))

TypeError: Cannot perform 'ror_' with a dtyped [object] array and scalar of type [bool]